# AI Mock Interviewer

**Samsung Innovation Campus — Coding & Programming | Chapter 5 Final Mini Project**

**Business problem this solves:** Fresh graduates preparing for campus placements need repeated interview practice, but mock interviews with friends or seniors are hard to schedule consistently. This project simulates a beginner job interview — it asks role-based questions, takes the candidate's answer, and uses Generative AI to give structured, honest feedback (similar to how a real interviewer would evaluate clarity, structure, and specificity).

**GenAI concepts applied:**
- Few-shot prompting (teaching the model what a strong vs weak answer looks like)
- Chain-of-Thought style evaluation (model reasons through the answer before scoring)
- A defined system persona (acts consistently as a professional but encouraging interviewer)
- Basic responsible-AI guardrail (keeps feedback constructive, not harsh)

**Author:** Abdul Khader | Presidency University, Bengaluru

## Step 1: Install and import the Gemini SDK

In [ ]:
!pip install -q google-generativeai

import google.generativeai as genai
from getpass import getpass

## Step 2: Set up the API key

Get a free Gemini API key from **https://aistudio.google.com/app/apikey**

Running the cell below will ask you to paste your key. It will NOT be saved in the notebook file, so it's safe to push this to GitHub.

In [ ]:
api_key = getpass("Enter your Gemini API key: ")
genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-1.5-flash")
print("Gemini model ready.")

## Step 3: Question bank

Organized by role, so the interviewer feels relevant to different beginner job tracks.

In [ ]:
question_bank = {
    "Software Developer": [
        "Tell me about yourself.",
        "Walk me through a project you built and your specific contribution.",
        "Why did you choose this tech stack for your project?",
        "Describe a time you were stuck on a bug. How did you solve it?",
        "Where do you see yourself in 3 years?"
    ],
    "HR / General": [
        "Tell me about yourself.",
        "What are your strengths and weaknesses?",
        "Why should we hire you?",
        "Describe a challenge you faced and how you handled it.",
        "Why do you want to work at our company?"
    ],
    "Data / Analyst": [
        "Tell me about yourself.",
        "Explain a project where you worked with data.",
        "How do you approach solving an ambiguous problem?",
        "What tools have you used for analysis, and why?",
        "Where do you see yourself in 3 years?"
    ]
}

## Step 4: The evaluation prompt (Few-shot + Chain-of-Thought)

This is the core GenAI technique in this project. We give the model:
1. A fixed persona (professional, encouraging interviewer)
2. Two worked examples (few-shot) of a strong and a weak answer
3. An instruction to reason step by step before giving a final verdict (Chain-of-Thought)

In [ ]:
def build_evaluation_prompt(question, answer):
    return f"""
You are a professional, encouraging job interviewer conducting a beginner-level mock interview.
Your job is to give constructive, honest feedback — never harsh, never sarcastic.

Here are two examples of how to evaluate an answer:

Example 1:
Question: "Tell me about yourself."
Answer: "I am good boy I like coding pls hire me."
Evaluation:
Reasoning: The answer lacks structure, has no specific details about education, skills, or
projects, and does not follow a professional tone.
Score: 3/10
Feedback: Structure your answer around education, key skills, and one project.
Avoid casual phrases like "pls hire me" — keep it confident and professional.

Example 2:
Question: "Tell me about yourself."
Answer: "I am a final year Computer Science student with a CGPA of 7.5. I have focused on
Python and Django, and built a Task Management web application handling authentication
and database design. I am looking to apply these skills in a full-stack developer role."
Evaluation:
Reasoning: The answer is structured (education, skills, project, goal), specific, and
professional in tone. It gives the interviewer concrete things to follow up on.
Score: 8/10
Feedback: Strong structure. Could be slightly improved by adding one sentence on why this
role specifically excites you.

Now evaluate this real answer the same way. Think step by step (Reasoning), then give a
Score out of 10, then give 2-3 lines of specific, constructive Feedback.

Question: "{question}"
Answer: "{answer}"

Respond in exactly this format:
Reasoning: <your reasoning>
Score: <x>/10
Feedback: <your feedback>
"""

## Step 5: Core interview function

In [ ]:
def get_feedback(question, answer):
    prompt = build_evaluation_prompt(question, answer)
    response = model.generate_content(prompt)
    return response.text

## Step 6: Main interview loop (menu-driven)

In [ ]:
def run_mock_interview():
    print("=" * 50)
    print("Welcome to the AI Mock Interviewer")
    print("=" * 50)

    roles = list(question_bank.keys())

    while True:
        print("\nChoose a role to practice for:")
        for i, role in enumerate(roles, start=1):
            print(f"{i}. {role}")
        print(f"{len(roles) + 1}. Exit")

        choice = input("\nEnter your choice: ").strip()

        if not choice.isdigit():
            print("Please enter a valid number.")
            continue

        choice = int(choice)

        if choice == len(roles) + 1:
            print("\nGood luck with your real interviews! Session ended.")
            break

        if choice < 1 or choice > len(roles):
            print("Invalid choice, try again.")
            continue

        selected_role = roles[choice - 1]
        questions = question_bank[selected_role]

        print(f"\nStarting mock interview for: {selected_role}")
        print("-" * 50)

        for q in questions:
            print(f"\nInterviewer: {q}")
            answer = input("Your answer: ")

            if answer.strip() == "":
                print("(No answer given — skipping feedback for this one.)")
                continue

            print("\nEvaluating your answer...\n")
            feedback = get_feedback(q, answer)
            print(feedback)
            print("-" * 50)

        again = input("\nPractice another role? (y/n): ").strip().lower()
        if again != "y":
            print("\nGood luck with your real interviews! Session ended.")
            break

## Step 7: Run it

In [ ]:
run_mock_interview()

## Reflection

**Which GenAI techniques were used, and why:**
- **Few-shot prompting** — two worked examples were given to the model so it learns the *pattern* of evaluation (structure, specificity, tone) rather than guessing what "good feedback" means.
- **Chain-of-Thought** — the model is asked to reason before scoring, which produces more consistent and explainable evaluations instead of a random number.
- **Persona / system framing** — the model is told to act as an encouraging, professional interviewer, which keeps the tone constructive rather than harsh, in line with Responsible AI principles from Chapter 4 (feedback should support the learner, not discourage them).

**Real-world extension:** this could be expanded with a larger question bank per company (TCS Digital, Accenture ASE style questions), voice input for spoken practice, or a score-tracking log across multiple sessions to show improvement over time.